In [122]:
import pandas as pd
import seaborn as sns
import plotly.express as px
from PIL import Image
import matplotlib.pyplot as plt
from pylab import rcParams
# Reglas de Asociación
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Análisis

In [123]:
df_items = pd.read_csv("https://raw.githubusercontent.com/pokengineer/DataScience/main/datasets/coffee_items.csv")
df_orders = pd.read_csv("https://raw.githubusercontent.com/pokengineer/DataScience/main/datasets/coffee_orders.csv")

In [124]:
df_items.head()

,item_id,sku,item_name,item_cat,item_size,item_price
0,It001,HDR-CAP-MD,Cappuccino,Hot Drinks,Medium,3.45
1,It002,HDR-CAP-LG,Cappuccino,Hot Drinks,Large,3.75
2,It003,HDR-LAT-MD,Latte,Hot Drinks,Medium,3.45
3,It004,HDR-LAT-LG,Latte,Hot Drinks,Large,3.75
4,It005,HDR-FLT,Flat White,Hot Drinks,NaN,3.15


In [125]:
df_orders.head()

,row_id,order_id,created_at,item_id,quantity,cust_name,in_or_out
0,1,ORD001,2024-02-12 07:04:19,It008,1,Alex,out
1,2,ORD002,2024-02-12 07:09:38,It014,1,Jordan,in
2,3,ORD003,2024-02-12 07:14:29,It008,1,Taylor,out
3,4,ORD004,2024-02-12 07:18:39,It019,1,Casey,out
4,5,ORD005,2024-02-12 07:23:44,It024,1,Jamie,out


In [126]:
print(df_orders.shape[0])

521


In [127]:
df_orders.order_id.duplicated().sum()

np.int64(91)

In [128]:
df_orders[df_orders['quantity']<=0]

,row_id,order_id,created_at,item_id,quantity,cust_name,in_or_out


# Columna de descripción

In [129]:
df_items['descripcion'] =df_items.apply(lambda x: str(x['item_name']) if x['item_size']!= x['item_size'] else  str(x['item_name']) + ' | ' + str(x['item_size']) , axis=1)

In [130]:
df_items.head(5)

,item_id,sku,item_name,item_cat,item_size,item_price,descripcion
0,It001,HDR-CAP-MD,Cappuccino,Hot Drinks,Medium,3.45,Cappuccino | Medium
1,It002,HDR-CAP-LG,Cappuccino,Hot Drinks,Large,3.75,Cappuccino | Large
2,It003,HDR-LAT-MD,Latte,Hot Drinks,Medium,3.45,Latte | Medium
3,It004,HDR-LAT-LG,Latte,Hot Drinks,Large,3.75,Latte | Large
4,It005,HDR-FLT,Flat White,Hot Drinks,NaN,3.15,Flat White


In [131]:
df_orders = pd.merge(df_orders,df_items[['item_id','descripcion']],on='item_id',how='left')

# Reglas de asociación

In [132]:
# Quitamos los ítems DOTCOM POSTAGE y POSTAGE porque no son productos.
# Llenamos con 0 los pedidos donde un producto no fué comprado
df_group = (df_orders.groupby(['order_id', 'descripcion'])['quantity'].sum().unstack().reset_index().fillna(0).set_index('order_id'))
# Seteamos True/False dependiendo de cada valor
df_group = df_group.applymap(lambda x: True if x >0 else False)
df_group

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_28708\2367238087.py:5: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_group = df_group.applymap(lambda x: True if x >0 else False)


descripcion,Cappuccino | Large,Cappuccino | Medium,Caramel Macchiato | Large,Caramel Macchiato | Medium,Cold Coffee | Large,Cold Coffee | Medium,Cold Mocha | Large,Cold Mocha | Medium,Espresso,Flat White,...,Latte | Large,Latte | Medium,Lemonade | Large,Lemonade | Medium,Mocha | Large,Mocha | Medium,Sandwich Ham&Cheese,Sandwich Salami&Mozzarella,White Mocha | Large,White Mocha | Medium
order_id,,,,,,,,,,,,,,,,,,,,,
ORD001,False,False,False,False,False,False,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False
ORD002,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
ORD003,False,False,False,False,False,False,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False
ORD004,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
ORD005,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ORD432,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
ORD433,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
ORD434,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [133]:
x = 0.005
frequent_itemsets = apriori(df_group, min_support=x, use_colnames=True)
frequent_itemsets.sort_values(by="support", ascending=False)

,support,itemsets
9,0.069409,(Flat White)
6,0.069409,(Cold Mocha | Large)
19,0.066838,(Mocha | Medium)
10,0.056555,(Hot Chocolate | Large)
23,0.056555,(White Mocha | Medium)
12,0.053985,(Iced Tea | Large)
16,0.053985,(Lemonade | Large)
1,0.053985,(Cappuccino | Medium)
22,0.051414,(White Mocha | Large)
7,0.051414,(Cold Mocha | Medium)


In [134]:
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)
# Ordenamos por confianza de mayor a menor
rules.sort_values(by="confidence", ascending=False).head(5)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
7,(Lemonade | Large),(Cold Mocha | Large),0.053985,0.069409,0.012853,0.238095,3.430335,1.0,0.009106,1.221401,0.748913,0.116279,0.181268,0.211640
23,(Sandwich Ham&Cheese),(White Mocha | Medium),0.041131,0.056555,0.007712,0.187500,3.315341,1.0,0.005386,1.161163,0.728329,0.085714,0.138794,0.161932
6,(Cold Mocha | Large),(Lemonade | Large),0.069409,0.053985,0.012853,0.185185,3.430335,1.0,0.009106,1.161019,0.761326,0.116279,0.138688,0.211640
13,(Espresso),(Sandwich Salami&Mozzarella),0.048843,0.048843,0.007712,0.157895,3.232687,1.0,0.005326,1.129499,0.726126,0.085714,0.114651,0.157895
12,(Sandwich Salami&Mozzarella),(Espresso),0.048843,0.048843,0.007712,0.157895,3.232687,1.0,0.005326,1.129499,0.726126,0.085714,0.114651,0.157895


# Consigna

## 1 - Conceptualmente que representa la métrica ``support`` y que representa la métrica ``confidence``.

### **Support (Soporte)**

- Representa la **frecuencia** con la que un conjunto de ítems aparece en el dataset.
- Es la proporción de transacciones que contienen ese conjunto de ítems.
- Fórmula: Support(X) = (Número de transacciones que contienen X) / (Total de transacciones)
- Ejemplo: Si el café con leche aparece en 100 de 1000 órdenes, su soporte es 0.10 (10%)

### **Confidence (Confianza)**

- Representa la **probabilidad condicional** de que ocurra el consecuente dado que ya ocurrió el antecedente.
- Mide qué tan fuerte es la relación entre dos conjuntos de ítems.
- Fórmula: Confidence(X → Y) = Support(X ∪ Y) / Support(X)
- Ejemplo: Si en el 80% de las veces que alguien compra café también compra azúcar, la confianza de la regla "café → azúcar" es 0.80 (80%)

## 2 - Obtener las reglas de asociacion usando item_name en lugar de la descripción.

Hacemos merge con ``item_name`` de ``df_items``.

In [135]:
df_orders_name = pd.merge(df_orders, df_items[['item_id','item_name']], on='item_id', how='left')

Agrupamos por ``order_id`` e ``item_name`` en lugar de descripción.

In [136]:
df_group_name = (df_orders_name.groupby(['order_id', 'item_name'])['quantity'].sum().unstack().reset_index().fillna(0).set_index('order_id'))

Convertimos a True/False

In [137]:
df_group_name = df_group_name.applymap(lambda x: True if x > 0 else False)
df_group_name.head()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_28708\586178754.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_group_name = df_group_name.applymap(lambda x: True if x > 0 else False)


item_name,Cappuccino,Caramel Macchiato,Cold Coffee,Cold Mocha,Espresso,Flat White,Hot Chocolate,Iced Tea,Latte,Lemonade,Mocha,Sandwich Ham&Cheese,Sandwich Salami&Mozzarella,White Mocha
order_id,,,,,,,,,,,,,,
ORD001,False,False,False,False,True,False,False,False,False,False,False,False,False,False
ORD002,False,False,False,False,False,False,True,False,False,False,False,False,False,False
ORD003,False,False,False,False,True,False,False,False,False,False,False,False,False,False
ORD004,False,False,False,False,False,False,False,True,False,False,False,False,False,False
ORD005,False,False,False,False,False,False,False,False,False,False,False,False,True,False


Aplicamos el algoritmo **Apriori** con el mismo soporte mínimo

In [138]:
frequent_itemsets_name = apriori(df_group_name, min_support=0.005, use_colnames=True)
frequent_itemsets_name.sort_values(by="support", ascending=False)

,support,itemsets
3,0.120823,(Cold Mocha)
13,0.107969,(White Mocha)
0,0.102828,(Cappuccino)
9,0.102828,(Lemonade)
10,0.097686,(Mocha)
8,0.095116,(Latte)
1,0.095116,(Caramel Macchiato)
7,0.092545,(Iced Tea)
6,0.089974,(Hot Chocolate)
2,0.077121,(Cold Coffee)


Generamos las reglas de asociación

In [139]:
rules_name = association_rules(frequent_itemsets_name, metric="lift", min_threshold=1)

Ordenamos por confianza de mayor a menor

In [140]:
rules_name.sort_values(by="confidence", ascending=False).head(10)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
15,(Sandwich Ham&Cheese),(White Mocha),0.041131,0.107969,0.007712,0.187500,1.736607,1.0,0.003271,1.097884,0.442359,0.054545,0.089157,0.129464
4,(Lemonade),(Cold Mocha),0.102828,0.120823,0.017995,0.175000,1.448404,1.0,0.005571,1.065670,0.345068,0.087500,0.061623,0.161968
8,(Sandwich Salami&Mozzarella),(Espresso),0.048843,0.048843,0.007712,0.157895,3.232687,1.0,0.005326,1.129499,0.726126,0.085714,0.114651,0.157895
9,(Espresso),(Sandwich Salami&Mozzarella),0.048843,0.048843,0.007712,0.157895,3.232687,1.0,0.005326,1.129499,0.726126,0.085714,0.114651,0.157895
5,(Cold Mocha),(Lemonade),0.120823,0.102828,0.017995,0.148936,1.448404,1.0,0.005571,1.054177,0.352130,0.087500,0.051393,0.161968
12,(Flat White),(White Mocha),0.069409,0.107969,0.010283,0.148148,1.372134,1.0,0.002789,1.047167,0.291436,0.061538,0.045042,0.121693
7,(Sandwich Ham&Cheese),(Cold Mocha),0.041131,0.120823,0.005141,0.125000,1.034574,1.0,0.000172,1.004774,0.034853,0.032787,0.004751,0.083777
11,(Flat White),(Iced Tea),0.069409,0.092545,0.007712,0.111111,1.200617,1.0,0.001289,1.020887,0.179558,0.050000,0.020460,0.097222
0,(Espresso),(Cold Coffee),0.048843,0.077121,0.005141,0.105263,1.364912,1.0,0.001375,1.031453,0.281081,0.042553,0.030494,0.085965
2,(Sandwich Salami&Mozzarella),(Cold Coffee),0.048843,0.077121,0.005141,0.105263,1.364912,1.0,0.001375,1.031453,0.281081,0.042553,0.030494,0.085965


## 3 - Obtener las reglas de asociacion del siguiente dataset

[Online Retail](http://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx). Pueden usar ``pd.read_excel()`` para archivos XLS o XLSX.

Cargamos el dataset de Online Retail

In [141]:
url = "http://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"
df_retail = pd.read_excel(url)
df_retail.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


Exploramos el dataset

In [142]:
print(f"Shape: {df_retail.shape}")
print(f"\nColumnas: {df_retail.columns.tolist()}")
print(f"\nValores nulos:")
print(df_retail.isnull().sum())

Shape: (541909, 8)

Columnas: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

Valores nulos:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


### Limpieza de datos

Eliminamos registros con valores nulos en CustomerID o Description

In [143]:
df_retail_clean = df_retail.dropna(subset=['CustomerID', 'Description'])

Eliminamos transacciones con cantidad negativa o cero (devoluciones o errores)

In [144]:
df_retail_clean = df_retail_clean[df_retail_clean['Quantity'] > 0]

Eliminamos facturas canceladas (las que empiezan con 'C')

In [145]:
df_retail_clean = df_retail_clean[~df_retail_clean['InvoiceNo'].astype(str).str.startswith('C')]
print(f"Registros después de limpieza: {df_retail_clean.shape[0]}")
df_retail_clean.head()

Registros después de limpieza: 397924


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


### Agrupamos por ``InvoiceNo`` (orden) y ``Description`` (producto)

Creamos la matriz de transacciones

In [146]:
df_retail_group = (df_retail_clean.groupby(['InvoiceNo', 'Description'])['Quantity'].sum().unstack().reset_index().fillna(0).set_index('InvoiceNo'))

In [147]:
df_retail_group = df_retail_group.applymap(lambda x: True if x > 0 else False)
print(f"Transacciones: {df_retail_group.shape[0]}")
print(f"Productos únicos: {df_retail_group.shape[1]}")
df_retail_group.head()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_28708\1466732615.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_retail_group = df_retail_group.applymap(lambda x: True if x > 0 else False)


Transacciones: 18536
Productos únicos: 3877


Description,4 PURPLE FLOCK DINNER CANDLES,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,I LOVE LONDON MINI RUCKSACK,NINE DRAWER OFFICE TIDY,OVAL WALL MIRROR DIAMANTE,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,...,ZINC STAR T-LIGHT HOLDER,ZINC SWEETHEART SOAP DISH,ZINC SWEETHEART WIRE LETTER RACK,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC TOP 2 DOOR WOODEN SHELF,ZINC WILLIE WINKIE CANDLE STICK,ZINC WIRE KITCHEN ORGANISER,ZINC WIRE SWEETHEART LETTER TRAY
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536365,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536366,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536367,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536368,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536369,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


### Aplicamos el algoritmo Apriori

Usamos un soporte mínimo de 0.01 (1%) - ajustable según necesidad

In [148]:
frequent_itemsets_retail = apriori(df_retail_group, min_support=0.01, use_colnames=True)
print(f"Conjuntos frecuentes encontrados: {len(frequent_itemsets_retail)}")
frequent_itemsets_retail.sort_values(by="support", ascending=False).head(10)

Conjuntos frecuentes encontrados: 973


,support,itemsets
587,0.106334,(WHITE HANGING HEART T-LIGHT HOLDER)
414,0.091929,(REGENCY CAKESTAND 3 TIER)
247,0.086319,(JUMBO BAG RED RETROSPOT)
338,0.074450,(PARTY BUNTING)
38,0.074180,(ASSORTED COLOUR BIRD ORNAMENT)
282,0.069486,(LUNCH BAG RED RETROSPOT)
468,0.061826,(SET OF 3 CAKE TINS PANTRY DESIGN )
374,0.059290,(POSTAGE)
274,0.056754,(LUNCH BAG BLACK SKULL.)
323,0.055514,(PACK OF 72 RETROSPOT CAKE CASES)


Generamos las reglas de asociación

In [149]:
rules_retail = association_rules(frequent_itemsets_retail, metric="lift", min_threshold=1)
print(f"Reglas de asociación encontradas: {len(rules_retail)}")

Reglas de asociación encontradas: 934


Top 10 reglas por confianza

In [150]:
rules_retail.sort_values(by="confidence", ascending=False).head(10)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
901,"(POPPY'S PLAYHOUSE LIVINGROOM , POPPY'S PLAYHO...",(POPPY'S PLAYHOUSE KITCHEN),0.011060,0.018666,0.010035,0.907317,48.607021,1.0,0.009828,10.588073,0.990380,0.509589,0.905554,0.722445
907,"(REGENCY CAKESTAND 3 TIER, PINK REGENCY TEACUP...",(GREEN REGENCY TEACUP AND SAUCER),0.014297,0.037279,0.012894,0.901887,24.193015,1.0,0.012361,9.812351,0.972570,0.333333,0.898088,0.623881
537,(REGENCY TEA PLATE PINK),(REGENCY TEA PLATE GREEN ),0.012085,0.014566,0.010898,0.901786,61.909259,1.0,0.010722,10.033507,0.995882,0.691781,0.900334,0.824967
620,"(PINK REGENCY TEACUP AND SAUCER, ROSES REGENCY...",(GREEN REGENCY TEACUP AND SAUCER),0.023522,0.037279,0.021040,0.894495,23.994742,1.0,0.020163,9.124923,0.981409,0.529172,0.890410,0.729447
906,"(REGENCY CAKESTAND 3 TIER, PINK REGENCY TEACUP...",(ROSES REGENCY TEACUP AND SAUCER ),0.014620,0.042242,0.012894,0.881919,20.877710,1.0,0.012276,8.111012,0.966228,0.293252,0.876711,0.593578
540,(REGENCY TEA PLATE PINK),(REGENCY TEA PLATE ROSES ),0.012085,0.017695,0.010628,0.879464,49.700457,1.0,0.010414,8.149491,0.991866,0.554930,0.877293,0.740037
612,"(REGENCY CAKESTAND 3 TIER, PINK REGENCY TEACUP...",(GREEN REGENCY TEACUP AND SAUCER),0.016670,0.037279,0.014620,0.877023,23.526037,1.0,0.013999,7.828443,0.973726,0.371742,0.872261,0.634604
900,"(POPPY'S PLAYHOUSE LIVINGROOM , POPPY'S PLAYHO...",(POPPY'S PLAYHOUSE BEDROOM ),0.011599,0.017048,0.010035,0.865116,50.746188,1.0,0.009837,7.287403,0.991798,0.539130,0.862777,0.726862
894,"(REGENCY CAKESTAND 3 TIER, PINK REGENCY TEACUP...",(ROSES REGENCY TEACUP AND SAUCER ),0.016670,0.042242,0.014297,0.857605,20.302132,1.0,0.013592,6.726072,0.966862,0.320435,0.851325,0.598024
506,(POPPY'S PLAYHOUSE LIVINGROOM ),(POPPY'S PLAYHOUSE KITCHEN),0.013595,0.018666,0.011599,0.853175,45.706487,1.0,0.011345,6.683678,0.991602,0.561358,0.850382,0.737281


### Ver reglas ordenadas por LIFT

Top 10 reglas por lift

In [151]:
rules_retail.sort_values(by="lift", ascending=False).head(10)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
537,(REGENCY TEA PLATE PINK),(REGENCY TEA PLATE GREEN ),0.012085,0.014566,0.010898,0.901786,61.909259,1.0,0.010722,10.033507,0.995882,0.691781,0.900334,0.824967
536,(REGENCY TEA PLATE GREEN ),(REGENCY TEA PLATE PINK),0.014566,0.012085,0.010898,0.748148,61.909259,1.0,0.010722,3.922605,0.998390,0.691781,0.745067,0.824967
902,"(POPPY'S PLAYHOUSE KITCHEN, POPPY'S PLAYHOUSE ...",(POPPY'S PLAYHOUSE LIVINGROOM ),0.013703,0.013595,0.010035,0.732283,53.863517,1.0,0.009848,3.684512,0.995070,0.581250,0.728594,0.735189
903,(POPPY'S PLAYHOUSE LIVINGROOM ),"(POPPY'S PLAYHOUSE KITCHEN, POPPY'S PLAYHOUSE ...",0.013595,0.013703,0.010035,0.738095,53.863517,1.0,0.009848,3.765861,0.994961,0.581250,0.734456,0.735189
530,(REGENCY MILK JUG PINK ),(REGENCY SUGAR BOWL GREEN),0.014674,0.014458,0.011114,0.757353,52.381694,1.0,0.010901,4.061626,0.995518,0.616766,0.753793,0.763005
531,(REGENCY SUGAR BOWL GREEN),(REGENCY MILK JUG PINK ),0.014458,0.014674,0.011114,0.768657,52.381694,1.0,0.010901,4.259150,0.995300,0.616766,0.765211,0.763005
900,"(POPPY'S PLAYHOUSE LIVINGROOM , POPPY'S PLAYHO...",(POPPY'S PLAYHOUSE BEDROOM ),0.011599,0.017048,0.010035,0.865116,50.746188,1.0,0.009837,7.287403,0.991798,0.539130,0.862777,0.726862
905,(POPPY'S PLAYHOUSE BEDROOM ),"(POPPY'S PLAYHOUSE LIVINGROOM , POPPY'S PLAYHO...",0.017048,0.011599,0.010035,0.588608,50.746188,1.0,0.009837,2.402575,0.997296,0.539130,0.583780,0.726862
541,(REGENCY TEA PLATE ROSES ),(REGENCY TEA PLATE PINK),0.017695,0.012085,0.010628,0.600610,49.700457,1.0,0.010414,2.473559,0.997531,0.554930,0.595724,0.740037
540,(REGENCY TEA PLATE PINK),(REGENCY TEA PLATE ROSES ),0.012085,0.017695,0.010628,0.879464,49.700457,1.0,0.010414,8.149491,0.991866,0.554930,0.877293,0.740037
